# First agent · Which tool can answer this question?

A library patron might ask how many copies of a title are available or where to find it.
Build a two-tool `smolagents.ToolCallingAgent` and inspect what happens when its environment changes.
The model interface uses a deterministic policy for these two question patterns. This makes routing
and errors reproducible in a local Python kernel with `smolagents==1.26.0`.

**Local practice** for the [Unit 1 first-agent lesson](https://huggingface.co/learn/agents-course/en/unit1/tutorial).

## Give each tool one job

The dictionary represents the **environment**. The functions are **tools** with named inputs
and observable results. The model will see those results through messages.

In [1]:
import re
from smolagents import Model, ToolCallingAgent, tool
from smolagents.models import ChatMessage, ChatMessageToolCall, ChatMessageToolCallFunction, MessageRole

catalogue = {"Night Atlas": {"copies": 3, "shelf": "Maps 2B"}}

@tool
def available_copies(title: str) -> str:
    """Look up the available copy count of one title.

    Args:
        title: The exact catalogue title.
    """
    if title not in catalogue:
        return f"Unknown title: {title}"
    return f"{title}: {catalogue[title]['copies']} copies available"

@tool
def find_shelf(title: str) -> str:
    """Look up the shelf location of one title.

    Args:
        title: The exact catalogue title.
    """
    if title not in catalogue:
        return f"Unknown title: {title}"
    return f"{title}: shelf {catalogue[title]['shelf']}"

print("Registered tool interfaces:", available_copies.name, find_shelf.name)

Registered tool interfaces: available_copies find_shelf


## Choose an action and read the reply

The policy chooses a tool from the user's wording. On the next step, it returns the observation
as its answer. A pretrained model can occupy this same interface. Here the rule's limits are
explicit, including what it does when the user leaves out the title.

In [2]:
class LibraryPolicy(Model):
    def generate(self, messages, **kwargs):
        latest = messages[-1]
        text = latest.content[0]["text"]
        if latest.role == MessageRole.TOOL_RESPONSE:
            name = "final_answer"
            arguments = {"answer": text.removeprefix("Observation:").strip()}
        else:
            title_match = re.search(r'"([^"]+)"', text)
            if title_match and "how many" in text.lower():
                name, arguments = "available_copies", {"title": title_match[1]}
            elif title_match and "where" in text.lower():
                name, arguments = "find_shelf", {"title": title_match[1]}
            else:
                name = "final_answer"
                arguments = {"answer": 'Ask where a title is or how many copies are available, and put the title in double quotes.'}
        action = ChatMessageToolCall(
            function=ChatMessageToolCallFunction(name=name, arguments=arguments),
            id=f"action-{len(messages)}", type="function",
        )
        return ChatMessage(role=MessageRole.ASSISTANT, content=f"Use {name}.", tool_calls=[action])

agent = ToolCallingAgent(tools=[available_copies, find_shelf], model=LibraryPolicy(),
                         max_steps=3, verbosity_level=-1)

def ask(question):
    answer = str(agent.run(question))
    steps = [step for step in agent.memory.steps if getattr(step, "tool_calls", None)]
    actions = [step.tool_calls[0].name for step in steps]
    return answer, actions

count_answer, count_actions = ask('How many copies of "Night Atlas" are available?')
print(count_actions, count_answer)
shelf_answer, shelf_actions = ask('Where can I find "Night Atlas"?')
print(shelf_actions, shelf_answer)
print("Message roles:", [m.role.value for m in agent.write_memory_to_messages()])

['available_copies', 'final_answer'] Night Atlas: 3 copies available
['find_shelf', 'final_answer'] Night Atlas: shelf Maps 2B
Message roles: ['system', 'user', 'assistant', 'tool-call', 'tool-response', 'assistant', 'tool-call', 'tool-response']


In [3]:
assert count_actions == ["available_copies", "final_answer"]
assert shelf_actions == ["find_shelf", "final_answer"]
assert count_answer == "Night Atlas: 3 copies available"
assert shelf_answer == "Night Atlas: shelf Maps 2B"

# A checkout changes the environment. The unchanged policy observes a new result.
catalogue["Night Atlas"]["copies"] = 2
updated, _ = ask('How many copies of "Night Atlas" are available?')
assert updated == "Night Atlas: 2 copies available"
print("After checkout:", updated)

missing, actions = ask('Where can I find "Not In Catalogue"?')
assert actions == ["find_shelf", "final_answer"] and missing == "Unknown title: Not In Catalogue"
print("Unknown title:", missing)
incomplete, actions = ask("Where is my book?")
assert actions == ["final_answer"] and "double quotes" in incomplete
print("Missing argument:", incomplete)

After checkout: Night Atlas: 2 copies available
Unknown title: Unknown title: Not In Catalogue
Missing argument: Ask where a title is or how many copies are available, and put the title in double quotes.


## Worked self-assessment

1. **Who chooses the tool?** The model policy maps “how many” to `available_copies` and “where”
   to `find_shelf`. The framework dispatches that structured request to the registered function.
2. **Which message supplies the shelf?** The user supplies the title. The tool response supplies
   “Maps 2B.” The assistant can then return that observation to the user. The system message
   supplies instructions and the available tool interfaces.
3. **Why did the count change?** The checkout changed the catalogue. A fresh tool invocation
   observed two copies. An agent that reused its earlier answer would have returned three.
4. **Why distinguish a missing title from an unknown title?** With no title argument, the policy
   can ask for clarification immediately. With a well-formed title absent from the catalogue, it
   learns that fact only after a lookup.

The [official Unit 1 final quiz](https://huggingface.co/spaces/agents-course/unit_1_quiz)
is the separate personal assessment for Hugging Face course credit.